# NCRB scanned tables: cloud OCR (unattended)

Reads every scanned page of NCRB's old reports (12,945 pages, mostly before 2000) with the best document-OCR model for these scans, and hands back the result for the [NCRB pipeline](https://github.com/reclaimchennai/NCRB).

**Runtime → Change runtime type → A100 GPU** (or L4), then run the two cells. Cell 1 asks once for access to your Google Drive (where everything is saved, so nothing is lost if the session resets) and starts the job in the background. Cell 2 shows progress, downloads the result zip when it is done and releases the GPU, so no compute units are used after that.

The job ([colab/run_all.py](https://github.com/reclaimchennai/NCRB/blob/main/colab/run_all.py)):
1. downloads the scanned pages ([`ocr-pages` release](https://github.com/reclaimchennai/NCRB/releases/tag/ocr-pages), 4.1 GB);
2. bake-off on 280 pages spread over every report and decade: OvisOCR2, PaddleOCR-VL-1.6 and GLM-OCR (whole-page models at the top of OmniDocBench v1.6, Oct 2026), then the best two again at a larger page size; the winner is the reading that reproduces the most of NCRB's own printed row and column totals;
3. the winner reads every page; crashes and stalls are restarted, the whole job stops at 14 hours.

If the session ends, run both cells again: pages already on Drive are skipped.

In [ ]:
# 1. Drive, code, vLLM, and the job in the background
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!test -d /content/NCRB && git -C /content/NCRB pull -q || git clone -q --depth 1 https://github.com/reclaimchennai/NCRB.git /content/NCRB
!pip install -q -U vllm pymupdf requests beautifulsoup4 lxml 2>&1 | tail -2
import torch, subprocess
DTYPE = 'auto' if torch.cuda.get_device_capability()[0] >= 8 else 'half'   # T4 has no bfloat16
# '[c]olab' so the check does not find its own command line
running = subprocess.run(['pgrep', '-f', '[c]olab/run_all.py'], capture_output=True).stdout.strip()
if not running:
    subprocess.Popen(f'cd /content/NCRB && nohup python colab/run_all.py --dtype {DTYPE} >> /content/drive/MyDrive/ncrb_ocr_run.log 2>&1 &', shell=True)
    import time; time.sleep(20)
    running = subprocess.run(['pgrep', '-f', '[c]olab/run_all.py'], capture_output=True).stdout.strip()
    print('job started' if running else 'job did not start; log:')
    if not running: print(open('/content/drive/MyDrive/ncrb_ocr_run.log').read()[-3000:])
else:
    print('job already running, pid', running.decode())

In [ ]:
# 2. progress; at the end the result zip downloads and the GPU is released
exec(open('/content/NCRB/colab/monitor.py').read())
watch()

On the laptop, with the zip in Downloads:

```
uv run python -m ncrb.vlm_import ~/Downloads/<model>.zip
uv run python -m ncrb.extract --scanned --vlm-ready --force
```